In [ ]:
#NAME: MADUGBA PRINCEWILL CHUKWUEMEKA
#STUDENT ID: X24297461

#Install packages
%pip install pymongo
#IMPORTS
import os
from dotenv import load_dotenv
import requests
import pandas as pd
from pymongo import MongoClient
from sqlalchemy import create_engine
from sqlalchemy.types import Integer, Float, String
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Loading the Shared environment
load_dotenv("../data.env", override=True)

MONGO_USERNAME = os.getenv("MONGO_USERNAME")
MONGO_PASSWORD = os.getenv("MONGO_PASSWORD")
MONGO_HOST = os.getenv("MONGO_HOST", "localhost")
MONGO_PORT = int(os.getenv("MONGO_PORT", "27017"))
MONGO_DB = os.getenv("MONGO_DB")
MONGO_COLLECTION = os.getenv("MONGO_COLLECTION")

In [ ]:
# Setting up Mongo Connection
MONGO_URI = os.getenv("MONGO_URI")
MONGO_DB  = os.getenv("MONGO_DB")
MONGO_COLLECTION = os.getenv("MONGO_COLLECTION")

# Checking the PostgreSQL credentials
#if not all([PG_USER, PG_PASS, PG_DB]):
    #raise RuntimeError("Missing PostgreSQL variables in data.env!")

# Fetching Helpers
def get_mongo_collection():
    client = MongoClient(
        host=MONGO_HOST,
        port=MONGO_PORT,
        username=MONGO_USERNAME,
        password=MONGO_PASSWORD,
        authSource="admin"
    )

    db = client[MONGO_DB]
    return db[MONGO_COLLECTION]

def get_postgres_engine(): #SQLAlchemy engine lets pandas write to Postgres easily using _sql()
    conn = (
        f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}"
    )
    return create_engine(conn)

In [ ]:
# Loading the shared environment
load_dotenv("../data.env", override=True)

PG_USER = os.getenv("DATABASE_USERNAME")
PG_PASS = os.getenv("DATABASE_PASSWORD")
PG_DB   = os.getenv("DATABASE_NAME")
PG_HOST = os.getenv("DATABASE_HOST")
PG_PORT = os.getenv("DATABASE_PORT")

In [ ]:
# Configurations for the World Bank API (what we are fetching and for what time ranges)

WB_BASE = "https://api.worldbank.org/v2/country/all/indicator"
YEARS = "2000:2023"

#This Mapping is useful when we want to label each record as primary/secondary and  male/female records based on their indicators
INDICATOR_META = {
    "SE.PRM.ENRR.FE": {"level":"primary","sex":"female"},
    "SE.PRM.ENRR.MA": {"level":"primary","sex":"male"},
    "SE.SEC.ENRR.FE": {"level":"secondary","sex":"female"},
    "SE.SEC.ENRR.MA": {"level":"secondary","sex":"male"},
}


def fetch_indicator(code):
    """Fetch JSON from World Bank API for one indicator."""
    #The parameters and request were built 
    url = f"{WB_BASE}/{code}"
    params = {"date": YEARS, "format": "json", "per_page": 20000} 

    resp = requests.get(url, params=params) #when a request is made and an error is spotted from the network/API it stop immediately
    resp.raise_for_status()

    data = resp.json() #The API will return a list of metadata, actual_data
    if len(data) < 2:
        return []

    return data[1]

In [ ]:
# Loading of RAW API to Mongo

def load_raw_to_mongo():
    coll = get_mongo_collection()
    

    coll.delete_many({}) #this is to avoid dupicate records

    total = 0
#Fetch each indicator, tag each record with the indicator_code
    for code in INDICATOR_META.keys():
        records = fetch_indicator(code)

        for r in records:
            r["indicator_code"] = code
        if records: #If only records are gotten back
            coll.insert_many(records)
            total += len(records)

    print(f"Inserted {total} raw records to MongoDB.")

In [ ]:
load_raw_to_mongo()

In [ ]:
# Converting Flatten Mongo to DataFrame
def flatten_mongo():
    #all this is to read the MongoDB raw docs and convert them into a tabular pandas DataFrame
    coll = get_mongo_collection()
    docs = list(coll.find({}))
    df = pd.json_normalize(docs)

    flat = pd.DataFrame({
        "country_code": df.get("countryiso3code"),
        "country_name": df.get("country.value"),
        "year": pd.to_numeric(df.get("date"), errors="coerce"),
        "indicator_code": df.get("indicator_code"),
        "indicator_name": df.get("indicator.value"),
        "value": pd.to_numeric(df.get("value"), errors="coerce"),
    })

    flat["level"] = flat["indicator_code"].map(lambda x: INDICATOR_META.get(x, {}).get("level"))
    flat["sex"] = flat["indicator_code"].map(lambda x: INDICATOR_META.get(x, {}).get("sex"))

    flat = flat.dropna(subset=["value", "year"])
    print(f"Flattened DataFrame shape: {flat.shape}")
    return flat

In [ ]:
flat=flatten_mongo()

In [ ]:
flat.head()

In [ ]:
# Analytical Pivot Table
def pivot_gender(df):
    pivot = df.pivot_table(
        index=["country_code", "country_name", "year"],
        columns=["level", "sex"],
        values="value"
    )
    pivot.columns = [f"{lvl}_{sex}" for (lvl, sex) in pivot.columns]
    pivot = pivot.reset_index()
    #Creating specific columns
    for col in ["primary_female", "primary_male", "secondary_female", "secondary_male"]:
        if col not in pivot.columns:
            pivot[col] = pd.NA
    pivot["primary_gap_female_minus_male"] = pivot["primary_female"] - pivot["primary_male"]
    pivot["secondary_gap_female_minus_male"] = pivot["secondary_female"] - pivot["secondary_male"]
    return pivot

In [ ]:
pivot_gender(flat)

In [ ]:
# Write to PostgreSQL
def write_to_postgres(df):
    engine = get_postgres_engine()

    dtypes = {
        "country_code": String(10),
        "country_name": String(255),
        "year": Integer(),
        "primary_female": Float(),
        "primary_male": Float(),
        "secondary_female": Float(),
        "secondary_male": Float(),
        "primary_gap_female_minus_male": Float(),
        "secondary_gap_female_minus_male": Float(),
    }

    df.to_sql("wb_gender_enrolment",engine,if_exists="replace",index=False,dtype=dtypes)

    print("wb_gender_enrolment written to PostgreSQL.")

# Main will be used to store the data

def main(): #All functions are called and are stored in the Prince ETL
    load_raw_to_mongo() 
    flat = flatten_mongo()
    pivoted = pivot_gender(flat)
    write_to_postgres(pivoted)
    print("Prince ETL complete.")

if __name__ == "__main__":
    main()

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# Load PostgreSQL connection details
load_dotenv("../data.env", override=True)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DATABASE_USERNAME"),
    password=os.getenv("DATABASE_PASSWORD"),
    host=os.getenv("DATABASE_HOST"),
    port=int(os.getenv("DATABASE_PORT")),
    database=os.getenv("DATABASE_NAME")
)

engine = create_engine(db_url)

# Load Prince's Nigeria enrolment data
prince_df = pd.read_sql("""
SELECT *
FROM wb_gender_enrolment
WHERE country_code = 'NGA'
ORDER BY year
""", engine)

print("Rows loaded:", len(prince_df))

# Visualisation folder
viz_dir = Path("../visualisations")
viz_dir.mkdir(exist_ok=True)


# Primary enrolment by sex
plt.figure(figsize=(10, 5))

plt.plot(
    prince_df["year"],
    prince_df["primary_female"],
    marker="o",
    label="Female"
)

plt.plot(
    prince_df["year"],
    prince_df["primary_male"],
    marker="o",
    label="Male"
)

plt.title("Nigeria - World Bank Primary Gross Enrolment Ratio by Sex")
plt.xlabel("Year")
plt.ylabel("Gross Enrolment Ratio (%)")
plt.legend()
plt.grid(True)

plt.savefig(
    viz_dir / "prince_primary_enrolment_by_sex_nigeria.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# Secondary enrolment by sex
plt.figure(figsize=(10, 5))

plt.plot(
    prince_df["year"],
    prince_df["secondary_female"],
    marker="o",
    label="Female"
)

plt.plot(
    prince_df["year"],
    prince_df["secondary_male"],
    marker="o",
    label="Male"
)

plt.title("Nigeria - World Bank Secondary Gross Enrolment Ratio by Sex")
plt.xlabel("Year")
plt.ylabel("Gross Enrolment Ratio (%)")
plt.legend()
plt.grid(True)

plt.savefig(
    viz_dir / "prince_secondary_enrolment_by_sex_nigeria.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


# Gender enrolment gap
plt.figure(figsize=(10, 5))

plt.plot(
    prince_df["year"],
    prince_df["primary_gap_female_minus_male"],
    marker="o",
    label="Primary"
)

plt.plot(
    prince_df["year"],
    prince_df["secondary_gap_female_minus_male"],
    marker="o",
    label="Secondary"
)

plt.axhline(0, linestyle="--")

plt.title("Nigeria - Gender Gap in World Bank Enrolment")
plt.xlabel("Year")
plt.ylabel("Gender Gap (Female - Male, percentage points)")
plt.legend()
plt.grid(True)

plt.savefig(
    viz_dir / "prince_gender_enrolment_gap_nigeria.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()